In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q fastapi uvicorn pyngrok python-multipart accelerate faiss-cpu langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters pypdf sentence-transformers transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 70.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, bu

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
).eval()


def generate_text(prompt, max_length=300):
    """Send the prompt to Mistral and return ONLY the new answer text."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=False,                      # stable, factual answers
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

# separator="\n" makes sure long pages are really cut into ~1000-character chunks
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=100)

# the PDF that is currently loaded
state = {"vectordb": None, "filename": None, "pages": 0, "chunks": 0}


def build_index(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    chunks = text_splitter.split_documents(documents)
    if not chunks:
        raise ValueError("No text found in this PDF (it may be a scanned image).")
    vectordb = FAISS.from_documents(chunks, embedding)
    return vectordb, len(documents), len(chunks)


def ask_question(query):
    if state["vectordb"] is None:
        raise ValueError("Please upload a PDF first.")
    docs = state["vectordb"].similarity_search(query, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""You are a helpful assistant. Use the following context to answer the question.
If the answer is not in the context, say that you could not find it in the document.

Context:
{context}

Question: {query}
Answer:"""

    answer = generate_text(prompt, max_length=300)
    sources = [
        {"page": int(d.metadata.get("page", 0)) + 1, "text": d.page_content[:300]}
        for d in docs
    ]
    return answer, sources

/tmp/ipykernel_49/364716640.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
NGROK_TOKEN = "3KNvrzhB4HBVGm3enkvCX4TEo4R_7FKbX9FXQK6koYZJMnLRB"
API_KEY = "secret123"

from fastapi import FastAPI, Request, HTTPException, UploadFile, File
from pydantic import BaseModel
import uvicorn, threading, time, socket, traceback, shutil
from pyngrok import ngrok, conf

ngrok.kill()  # close any old tunnel first

app = FastAPI()
lock = threading.Lock()   # one request at a time on the GPU


class Question(BaseModel):
    question: str


def check_auth(req):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")


@app.get("/health")
def health():
    return {"ok": True, "pdf": state["filename"]}


@app.post("/upload")
def upload(request: Request, file: UploadFile = File(...)):
    check_auth(request)
    path = "/kaggle/working/uploaded.pdf"
    with open(path, "wb") as f:
        shutil.copyfileobj(file.file, f)
    try:
        with lock:
            t = time.time()
            vectordb, pages, n_chunks = build_index(path)
            state.update(vectordb=vectordb, filename=file.filename, pages=pages, chunks=n_chunks)
            print(f"Indexed {file.filename}: {pages} pages, {n_chunks} chunks in {time.time() - t:.0f}s")
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=400, detail=str(e))
    return {"filename": file.filename, "pages": pages, "chunks": n_chunks}


@app.post("/ask")
def ask(request: Request, body: Question):
    check_auth(request)
    if not body.question.strip():
        raise HTTPException(status_code=400, detail="Empty question")
    try:
        with lock:
            t = time.time()
            answer, sources = ask_question(body.question)
            print(f"Answered in {time.time() - t:.0f}s")
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=400, detail=str(e))
    return {"answer": answer, "sources": sources}


def free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port


port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)


def run():
    uvicorn.run(app, host="0.0.0.0", port=port)


threading.Thread(target=run, daemon=True).start()
time.sleep(2)

# local test (inside Kaggle, no ngrok involved)
import requests
print(requests.get(f"http://127.0.0.1:{port}/health").json())


Your public URL: https://vineyard-evade-tarantula.ngrok-free.dev                                    


INFO:     Started server process [49]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:34959 (Press CTRL+C to quit)


INFO:     127.0.0.1:52346 - "GET /health HTTP/1.1" 200 OK
{'ok': True, 'pdf': None}
Indexed The Lighthouse Keeper.pdf: 1 pages, 4 chunks in 2s
INFO:     156.204.38.217:0 - "POST /upload HTTP/1.1" 200 OK
Answered in 4s
INFO:     156.204.38.217:0 - "POST /ask HTTP/1.1" 200 OK
